In [1]:
!pip install nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
import os

os.listdir()

['.config', 'resume_dataset.zip', 'sample_data']

In [5]:
import zipfile

with zipfile.ZipFile("resume_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("resume_dataset")

In [6]:
import os

os.listdir("resume_dataset")

['Resume', 'data']

In [7]:
import os

os.listdir("resume_dataset/data")

['data']

In [11]:
import os

os.listdir("resume_dataset/data/data")

['HR',
 'AVIATION',
 'CONSULTANT',
 'DESIGNER',
 'FINANCE',
 'CONSTRUCTION',
 'BANKING',
 'PUBLIC-RELATIONS',
 'APPAREL',
 'DIGITAL-MEDIA',
 'CHEF',
 'AUTOMOBILE',
 'AGRICULTURE',
 'BPO',
 'SALES',
 'ARTS',
 'ACCOUNTANT',
 'ENGINEERING',
 'TEACHER',
 'HEALTHCARE',
 'INFORMATION-TECHNOLOGY',
 'ADVOCATE',
 'BUSINESS-DEVELOPMENT',
 'FITNESS']

In [12]:
import os

path = "resume_dataset/data/data/INFORMATION-TECHNOLOGY"

files = os.listdir(path)

print("Total Files:", len(files))
print(files[:5])

Total Files: 120
['10839851.pdf', '20001721.pdf', '26746496.pdf', '35325329.pdf', '10265057.pdf']


In [13]:
!pip install pdfplumber
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [14]:
path = "resume_dataset/data/data/INFORMATION-TECHNOLOGY"

files = os.listdir(path)

print("Total Files:", len(files))
print(files[:5])

Total Files: 120
['10839851.pdf', '20001721.pdf', '26746496.pdf', '35325329.pdf', '10265057.pdf']


In [15]:
import pdfplumber

pdf_path = "resume_dataset/data/data/INFORMATION-TECHNOLOGY/10839851.pdf"

with pdfplumber.open(pdf_path) as pdf:
    text = ""

    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text

print(text[:3000])

INFORMATION TECHNOLOGY SPECIALIST
Professional Summary
Seeking to obtain a career in Information Assurance with a focus on Cyber Network Defense
Seeking to obtain a career in Information Assurance with a focus on Cyber Network Defense
Seeking to obtain a career in Information Assurance with a focus on Cyber Network Defense
Skills
Desktops,
Ethernet cables
Cisco routers
Video & Sound Cards
CD-ROM Drives
Multiplexors
Scanners
Monitors
Switches
TCP/IP Configuration
Installing, adding and deleting user accounts with Active Directory
Strong software and application knowledge such as Avaya, Microsoft Office, and Remedy
Experience with Information Technology Service Management (ITSM)
Desktops,
Ethernet cables
Cisco routers
Video & Sound Cards
CD-ROM Drives
Multiplexors
Scanners Experience with Information TechnologyÂ
Monitors Strong Â software and application knowledge such as
Switches Avaya,Microsoft Office,and Remedy
TCP/IP Configuration Installing,adding and deleting user accounts with Act

In [1]:
import os

total = 0

for category in os.listdir("resume_dataset/data/data"):
    category_path = os.path.join("resume_dataset/data/data", category)

    if os.path.isdir(category_path):
        total += len([
            f for f in os.listdir(category_path)
            if f.endswith(".pdf")
        ])

print("Total PDFs:", total)

Total PDFs: 2484


In [2]:
import os

for category in os.listdir("resume_dataset/data/data"):
    category_path = os.path.join("resume_dataset/data/data", category)

    if os.path.isdir(category_path):
        pdf_count = len([
            f for f in os.listdir(category_path)
            if f.endswith(".pdf")
        ])

        print(category, ":", pdf_count)

HR : 110
AVIATION : 117
CONSULTANT : 115
DESIGNER : 107
FINANCE : 118
CONSTRUCTION : 112
BANKING : 115
PUBLIC-RELATIONS : 111
APPAREL : 97
DIGITAL-MEDIA : 96
CHEF : 118
AUTOMOBILE : 36
AGRICULTURE : 63
BPO : 22
SALES : 116
ARTS : 103
ACCOUNTANT : 118
ENGINEERING : 118
TEACHER : 102
HEALTHCARE : 115
INFORMATION-TECHNOLOGY : 120
ADVOCATE : 118
BUSINESS-DEVELOPMENT : 120
FITNESS : 117


In [2]:
!pip install tqdm

import os
import pdfplumber
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

base_path = "resume_dataset/data/data"

pdf_files = []

for category in os.listdir(base_path):
    category_path = os.path.join(base_path, category)

    if os.path.isdir(category_path):
        for file in os.listdir(category_path):
            if file.endswith(".pdf"):
                pdf_files.append(
                    (os.path.join(category_path, file), category)
                )

print("Total PDFs:", len(pdf_files))


def extract_pdf(args):
    pdf_path, category = args

    try:
        text = ""

        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()

                if page_text:
                    text += page_text

        return [text, category]

    except:
        return None


with ProcessPoolExecutor() as executor:
    results = list(
        tqdm(
            executor.map(extract_pdf, pdf_files),
            total=len(pdf_files)
        )
    )

results = [r for r in results if r is not None]

df = pd.DataFrame(results, columns=["Resume", "Category"])

print("Final Shape:", df.shape)

df.head()

Total PDFs: 2484


100%|██████████| 2484/2484 [25:33<00:00,  1.62it/s]


Final Shape: (2484, 2)


,Resume,Category
0,"SENIOR HR MANAGER, HR BUSINESS PARTNER\nSummar...",HR
1,HR ASSOCIATE MOBILIZATION COORDINATOR\nSummary...,HR
2,HR INTERN\nSummary\nHighly driven Recruiter wh...,HR
3,HR ASSISTANT III\nCertifications\nJohn A. Loga...,HR
4,"HR SPECIALIST\nSummary\nDedicated, Driven, and...",HR


In [3]:
df.to_csv("resume_text_dataset.csv", index=False)

In [4]:
from google.colab import files
files.download("resume_text_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
print(df.shape)

(2484, 2)


In [6]:
df["Category"].value_counts()

,count
Category,
BUSINESS-DEVELOPMENT,120
INFORMATION-TECHNOLOGY,120
FINANCE,118
CHEF,118
ACCOUNTANT,118
ENGINEERING,118
ADVOCATE,118
AVIATION,117
FITNESS,117


In [8]:
print(df["Category"].unique())

['HR' 'AVIATION' 'CONSULTANT' 'DESIGNER' 'FINANCE' 'CONSTRUCTION'
 'BANKING' 'PUBLIC-RELATIONS' 'APPAREL' 'DIGITAL-MEDIA' 'CHEF'
 'AUTOMOBILE' 'AGRICULTURE' 'BPO' 'SALES' 'ARTS' 'ACCOUNTANT'
 'ENGINEERING' 'TEACHER' 'HEALTHCARE' 'INFORMATION-TECHNOLOGY' 'ADVOCATE'
 'BUSINESS-DEVELOPMENT' 'FITNESS']


In [9]:
import re
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_resume(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df["Cleaned_Resume"] = df["Resume"].apply(clean_resume)

df[["Category", "Cleaned_Resume"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Category,Cleaned_Resume
0,HR,senior hr manager hr business partner summary ...
1,HR,hr associate mobilization coordinator summary ...
2,HR,hr intern summary highly driven recruiter main...
3,HR,hr assistant iii certifications john logan col...
4,HR,hr specialist summary dedicated driven dynamic...


In [10]:
print(df["Cleaned_Resume"][0][:1000])

senior hr manager hr business partner summary highly dedicated accomplished human resources manager record proficiency employee relations training development programs recruitment boarding payroll management benefits administration hrms database administration job description development wage salary reviews record keeping compliance proven leader championing company values vision expectations effective communication facilitation aligns hr strategy business objectives assesses anticipates hr related needs communicates proactively within global hr teams management seeks develop highly effective integrated hr solutions experience senior hr manager hr business partner january january company name city state hr manager january january quality service manager hr manager january january company name city state transferred peo model full service payroll benefits set including rfp various payroll benefits vendors interviewing best fit completing implementation phase working follow issues worker

In [11]:
from sklearn.preprocessing import LabelEncoder

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
from sklearn.linear_model import LogisticRegression

In [15]:
from sklearn.metrics import accuracy_score

In [16]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df["Category"])

print("Number of Classes:", len(encoder.classes_))
print(encoder.classes_)

Number of Classes: 24
['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df["Cleaned_Resume"])

print(X.shape)

(2484, 10000)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(1987, 10000)
(497, 10000)


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=3000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
}

results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    results[name] = accuracy

    print(f"{name} Accuracy: {accuracy:.4f}")

print("\nFinal Results")
print(results)


Training Logistic Regression...
Logistic Regression Accuracy: 0.6680

Training Linear SVM...
Linear SVM Accuracy: 0.7485

Training Random Forest...
Random Forest Accuracy: 0.7726

Final Results
{'Logistic Regression': 0.6680080482897385, 'Linear SVM': 0.7484909456740443, 'Random Forest': 0.772635814889336}


In [20]:
best_model_name = max(results, key=results.get)

print("Best Model:", best_model_name)
print("Accuracy:", results[best_model_name])

Best Model: Random Forest
Accuracy: 0.772635814889336


In [21]:
print(results)

{'Logistic Regression': 0.6680080482897385, 'Linear SVM': 0.7484909456740443, 'Random Forest': 0.772635814889336}


In [22]:
!pip install xgboost

In [27]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)
xgb.fit(X_train, y_train)

pred = xgb.predict(X_test)

acc = accuracy_score(y_test, pred)

print("XGBoost Accuracy:", acc)

XGBoost Accuracy: 0.8088531187122736


In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X = tfidf.fit_transform(df["Cleaned_Resume"])

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB(alpha=0.1)

nb.fit(X_train, y_train)

pred = nb.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Naive Bayes Accuracy:", acc)

Naive Bayes Accuracy: 0.5613682092555332


In [33]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score

et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)

pred = et.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Extra Trees Accuracy:", acc)

Extra Trees Accuracy: 0.7525150905432596


In [34]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    C=5,
    max_iter=5000,
    solver="lbfgs",
    n_jobs=-1
)

lr.fit(X_train, y_train)

pred = lr.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Tuned Logistic Regression:", acc)

Tuned Logistic Regression: 0.6901408450704225


In [35]:
import joblib

joblib.dump(xgb, "resume_model.pkl")
joblib.dump(tfidf, "tfidf.pkl")
joblib.dump(encoder, "label_encoder.pkl")

print("All files saved successfully!")

All files saved successfully!


In [36]:
from google.colab import files

files.download("resume_model.pkl")
files.download("tfidf.pkl")
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>